## 5. Breakdown: when the Krylov space stops growing (`gmres_v4`)

*Same math, computed safely.* Like Steps 2–4, this step doesn't change what GMRES computes. It changes what the solver does, and what it reports, when the Krylov space stops growing. Wherever that doesn't happen, the new solver produces exactly the same iterates as `gmres_v3`, and we'll check that it does.

**Where Step 4 left off.** We want to solve $Ax = b$ for an $n \times n$ matrix $A$ that we can only apply to vectors, through a function `matvec` $: v \mapsto Av$. Starting from $x_0 = 0$, GMRES picks at step $k$ the vector $x_k$ in the Krylov space

$$
\mathcal{K}_k = \operatorname{span}\{\, b,\ Ab,\ \dots,\ A^{k-1} b \,\}
$$

whose residual $r_k = b - A x_k$ has the smallest 2-norm, and stops once $\|r_k\|_2 \le \max(\text{rtol} \cdot \|b\|_2,\ \text{atol})$. Three pieces from Steps 2–4 do the work:

- **Arnoldi (Step 2)** builds an orthonormal basis $v_1, v_2, \dots$ of $\mathcal{K}_k$, one matvec per step, starting from $v_1 = b / \beta$ with $\beta = \|b\|_2$. Step $k$ applies $A$ to $v_k$ and removes its components $h_{1k}, \dots, h_{kk}$ along $v_1, \dots, v_k$. What is left has length $h_{k+1,k}$, and dividing it by that length gives $v_{k+1}$.
- **The Arnoldi relation (Step 3).** The numbers $h_{ij}$ fill the $(k+1) \times k$ upper Hessenberg matrix $\bar H_k$, and $A V_k = V_{k+1} \bar H_k$, where $V_k = [\, v_1 \mid \cdots \mid v_k \,]$. We write $H_k$ for the top $k \times k$ block of $\bar H_k$, that is $\bar H_k$ without its last row. GMRES then becomes the small problem $\min_y \|\beta e_1 - \bar H_k\, y\|_2$, with $e_1 = (1, 0, \dots, 0) \in \mathbb{R}^{k+1}$ and $x_k = V_k\, y$.
- **Givens rotations (Step 4)** turn $\bar H_k$ into an upper-triangular matrix, one rotation per column. The rotation $G_k$ for column $k$ is built from two numbers: $\tilde h_{kk}$, the diagonal entry of column $k$ after the earlier rotations $G_1, \dots, G_{k-1}$, and the subdiagonal entry $h_{k+1,k}$. Its cosine and sine are
  $$
  c_k = \frac{\tilde h_{kk}}{R_{kk}}, \qquad s_k = \frac{h_{k+1,k}}{R_{kk}}, \qquad R_{kk} = \sqrt{\tilde h_{kk}^2 + h_{k+1,k}^2} ,
  $$
  and $R_{kk}$ becomes the $k$-th diagonal entry, the **pivot**, of the triangular factor $R_k$. The same rotations turn $\beta e_1$ into $g = (g_1, \dots, g_{k+1})$. Then $\|r_k\|_2 = |g_{k+1}| = |s_k|\, \|r_{k-1}\|_2$, and $y$ solves $R_k\, y = g_{1:k}$ by back-substitution, where $g_{1:k} = (g_1, \dots, g_k)$.

We call $(\tilde h_{kk},\ h_{k+1,k})$ the **rotation pair** of step $k$. This step is about what that pair looks like when the Krylov space stops growing.

**What Step 4 assumed.** Step 4 found that the residual drops to zero exactly when $s_k = 0$, which happens when Arnoldi finds no new direction, $h_{k+1,k} = 0$. That is a **breakdown**. Steps 2–4 met several (`geo3` at $k = 3$, `diag_spread` and `convdiff_n` at $k = 20$, `convdiff_n` with $c = 0$ at $k = 10$), and each time the answer was exact. But the conclusion rested on a parenthetical in section 4.1: $R_k$ has to be invertible, so that $R_k\, y = g_{1:k}$ can be solved. Section 4.1 justified that "as long as no $h_{j+1,j}$ is zero", which is exactly what a breakdown breaks.

**The plan for this step.** Run `gmres_v3` on small matrices where $R_k$ is *not* invertible at the breakdown, and watch it report success with a useless answer (5.1). Explain when a breakdown happens and what makes it good or bad (5.2), go through the good ones (5.3) and the bad ones (5.4), and collect which kinds of matrix give which (5.5). Then look at how a breakdown is detected in floating point (5.6), and build `gmres_v4`, which keeps production's two safety checks and reports a breakdown in its record (5.7, 5.8). Last, a one-line change to one of those checks that does better on one example (5.9).

### Setup

[tutorial_utils.py](tutorial_utils.py) holds what every step of the tutorial uses: `CountingMatvec` from Step 1, the example problems and the plot style. Each example builder returns a pair `(A, b)`. As before, a solver only ever gets the matrix wrapped in a `CountingMatvec`; we use `A` directly only to *check* answers.

The example problems in this notebook, each a matrix $A$ with a right-hand side $b$:

- **`eig_rhs`** (new in this step): an upper-triangular 3×3,
  $$
  A = \begin{pmatrix} 2 & 1 & 0 \\ 0 & 3 & 1 \\ 0 & 0 & 4 \end{pmatrix}, \qquad b = e_1 = (1, 0, 0) .
  $$
  Its eigenvalues are its diagonal entries 2, 3 and 4, and $b$ is an eigenvector: $Ab = 2b$.
- **`singular3`** (new): $A = \operatorname{diag}(1, 2, 0)$ with $b = (1, 1, 1)$. $A$ squashes every vector onto the $xy$-plane (it sets the third entry to zero), so $Ax = b$ has no solution: nothing reaches $b$'s third entry. Called as `singular3(rhs="in_range")`, the same $A$ comes with $b = (1, 1, 0)$, which does lie in the $xy$-plane, and $x = (1, \tfrac12, 0)$ solves $Ax = b$.
- **`nilpotent2`** (new):
  $$
  A = \begin{pmatrix} 0 & 1 \\ 0 & 0 \end{pmatrix}, \qquad b = e_1 = (1, 0) .
  $$
  $Ab = 0$, yet $Ax = b$ has a solution, $x = e_2 = (0, 1)$. ($A^2 = 0$, which makes $A$ *nilpotent*.)
- **A matrix with repeated eigenvalues** (new, built in this notebook, not in `tutorial_utils.py`): $A = \operatorname{diag}(1, 1, 2, 2, 3, 3)$ with $b = (1, \dots, 1) \in \mathbb{R}^6$.
- **`geo3`**: the 3×3 running example from Steps 1–4,
  $$
  A = \begin{pmatrix} 3 & 1 & 0 \\ 0 & 2 & 1 \\ 1 & 0 & 2 \end{pmatrix}, \qquad b = (1, 1, 1) ,
  $$
  nonsymmetric and well conditioned, with the exact solution $x = (3, 4, 5)/13$.
- **`diag_narrow`**: the 100×100 diagonal matrix with eigenvalues evenly spaced from 1 to 1.5, $\lambda_i = 1 + 0.5\,(i-1)/99$, and $b = (1, \dots, 1)$. GMRES solves it in 11 steps, without a breakdown.
- **`diag_spread`**: the 20×20 diagonal matrix with eigenvalues evenly spaced on a log scale from 1 to $10^4$, $\lambda_i = 10^{4(i-1)/19}$, and $b = (1, \dots, 1)$. Section 5.9 also uses its 12-decade version, $\lambda_i = 10^{12(i-1)/19}$ (condition number $10^{12}$).
- **`convdiff_n`**: the 20×20 tridiagonal matrix from Step 3, section 3.3, which discretises $-u''(s) + c\, u'(s)$ on $[0, 1]$ with $u(0) = u(1) = 0$, on the 20 interior grid points $s_i = i\, \Delta s$ with spacing $\Delta s = 1/21$:
  $$
  \frac{2}{\Delta s^2} \text{ on the diagonal}, \qquad
  -\frac{1}{\Delta s^2} - \frac{c}{2 \Delta s} \text{ below it}, \qquad
  -\frac{1}{\Delta s^2} + \frac{c}{2 \Delta s} \text{ above it}.
  $$
  With velocity $c = 20$ and $b = (1, \dots, 1)$ it is nonsymmetric. With $c = 0$ it is the symmetric matrix of $-u''$ alone, used here with two right-hand sides: $b = (1, \dots, 1)$, and $b_i = \sin(\pi s_i)$. As in Step 4, the cosines $c_k$ of the rotations always carry a subscript.
- **`shift_n`**: the 8×8 cyclic shift from Step 4, $A e_j = e_{j+1}$ and $A e_8 = e_1$, with $b = e_1$, where $e_j$ is the $j$-th unit vector.

`%autoreload 2` picks up edits to `tutorial_utils.py` without restarting the kernel.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np  # noqa: I001
import matplotlib.pyplot as plt

from tutorial_utils import (
    # the matvec wrapper from Step 1
    CountingMatvec,
    # loading definitions from an earlier step's notebook
    import_from_notebook,
    # example problems; each returns (A, b)
    geo3, diag_narrow, diag_spread, convdiff_n, shift_n, eig_rhs, singular3, nilpotent2,
    # checking one solver version against another
    compare_versions,
    # plot style
    style_axes, plot_residuals, plot_matrix, add_figure_caption, REFERENCE_COLOR, SERIES_COLORS,
)

np.set_printoptions(precision=4, suppress=True)
eps = np.finfo(float).eps   # machine epsilon, about 2.2e-16

A_eig, b_eig = eig_rhs()                               # 3 x 3, b an eigenvector (new)
A_sing, b_sing = singular3()                           # diag(1, 2, 0), b = (1, 1, 1) (new)
_, b_in_range = singular3(rhs="in_range")              # the same A, b = (1, 1, 0) (new)
A_nil, b_nil = nilpotent2()                            # 2 x 2, A b = 0 (new)
A_repeated = np.diag([1.0, 1.0, 2.0, 2.0, 3.0, 3.0])   # repeated eigenvalues (new, built here)
b_repeated = np.ones(6)
A_geo3, b_geo3 = geo3()                                # 3 x 3, the running example
A_narrow, b_narrow = diag_narrow()                     # 100 x 100 diagonal, eigenvalues in [1, 1.5]
A_spread, b_spread = diag_spread()                     # 20 x 20 diagonal, eigenvalues from 1 to 1e4
A_spread12, b_spread12 = diag_spread(20, decades=12)   # 20 x 20 diagonal, eigenvalues from 1 to 1e12
A_cd, b_cd = convdiff_n(20, c=20.0)                    # 20 x 20 tridiagonal, nonsymmetric
A_cd0, b_cd0 = convdiff_n(20, c=0.0)                   # 20 x 20 tridiagonal, symmetric, b = ones
_, b_cd0_sine = convdiff_n(20, c=0.0, rhs="sine")      # the same A, b_i = sin(pi s_i)
A_shift, b_shift = shift_n(8)                          # 8 x 8 cyclic shift, b = e_1

**From Steps 2 and 4.** From Step 4 we need `gmres_v3`, the solver this step makes safe, with its `givens_rotation` and `back_substitution`. From Step 2 we need the three Arnoldi step functions, which fill $v_{k+1}$ and return column $k$ of $\bar H$ together with a breakdown flag, and `arnoldi`, which runs $k$ steps and returns $V_{k+1}$, $\bar H_k$ and $\beta$ (stopping early at a breakdown). `import_from_notebook` loads them straight from [GMRES_4_givens.ipynb](GMRES_4_givens.ipynb) and [GMRES_2_arnoldi_orthogonalise.ipynb](GMRES_2_arnoldi_orthogonalise.ipynb), running only those notebooks' imports and function definitions. To read any of them here, run `gmres_v3??` and so on.

In [ ]:
# Built in Steps 2 and 4. Loaded from those notebooks, so each stays the only copy.
gmres_v3, givens_rotation, back_substitution = import_from_notebook(
    "GMRES_4_givens.ipynb", "gmres_v3", "givens_rotation", "back_substitution")
(arnoldi_step_classical_gram_schmidt_twice, arnoldi_step_classical_gram_schmidt,
 arnoldi_step_modified_gram_schmidt, arnoldi) = import_from_notebook(
    "GMRES_2_arnoldi_orthogonalise.ipynb", "arnoldi_step_classical_gram_schmidt_twice",
    "arnoldi_step_classical_gram_schmidt", "arnoldi_step_modified_gram_schmidt", "arnoldi")

### 5.1 `gmres_v3` on three small matrices

Three examples, each with a breakdown, run through `gmres_v3` with its default tolerance, $\text{rtol} = 10^{-10}$. For each one we print what the solver reports (its `converged` flag and the last entry of `residual_history`, which is $|g_{k+1}|$), the $x$ it returns, and the true residual $\|b - Ax\|_2$ of that $x$, measured using $A$ directly. On `nilpotent2`, `gmres_v3` divides by zero; NumPy's warning about it is switched off so that the output stays readable.

In [ ]:
for name, A, b in [("eig_rhs", A_eig, b_eig), ("singular3", A_sing, b_sing), ("nilpotent2", A_nil, b_nil)]:
    with np.errstate(divide="ignore", invalid="ignore"):   # gmres_v3 divides by zero on nilpotent2
        x, record = gmres_v3(CountingMatvec(A), b)
        true_residual = np.linalg.norm(b - A @ x)
    print(f"{name}:")
    print(f"  gmres_v3 reports  converged = {record['converged']},  ||r_k|| = |g_k+1| = {record['residual_history'][-1]:.1f}"
          f"  after {record['iterations']} steps")
    print(f"  residual history  {np.array(record['residual_history'])}")
    print(f"  it returns        x = {x}")
    print(f"  true residual     ||b - A x|| = {true_residual:.3g}    (||b|| = {np.linalg.norm(b):.3g})")

All three report convergence with a residual of exactly 0. Only the first is telling the truth.

- **`eig_rhs`**: $x = (0.5, 0, 0) = b/2$, the exact solution, after one step.
- **`singular3`**: the residual goes $1.73 \to 1.10 \to 1.00 \to 0$, and the returned $x_3$ has a third entry of $-1.47 \times 10^{16}$. Its true residual is 2.24, *larger* than the $\|b\|_2 = 1.73$ of the starting guess $x_0 = 0$.
- **`nilpotent2`**: $x = (\infty, \text{NaN})$, and its true residual is NaN.

The reported zero comes from the same place each time. At a breakdown, Arnoldi stores $h_{k+1,k} = 0$, so $s_k = 0$ and $g_{k+1} = -s_k\, g_k = 0$. Reading the residual off as $|g_{k+1}|$ assumes that $R_k\, y = g_{1:k}$ can be solved, and on `singular3` and `nilpotent2` it can't. Sections 5.2–5.4 explain why.

This matters beyond a wrong number. The `converged` flag is all a caller checks. In the simulation the caller is a Newton iteration, which adds the GMRES solution to its state as a correction, and it does so even when GMRES says it failed: production's `newton_krylov.py` notes that "the correction of a failed linear solve is still applied; the outer test judges it." A correction of $10^{16}$, or of NaN, would wreck the state with no sign of where it came from.

### 5.2 What a breakdown is, and when it happens

**What it is.** A breakdown at step $k$ means $h_{k+1,k} = 0$: once its components along $v_1, \dots, v_k$ are removed, nothing is left of $A v_k$, so $A v_k$ already lies in $\mathcal{K}_k$. Every earlier $A v_j$, $j < k$, lies in $\mathcal{K}_{j+1} \subseteq \mathcal{K}_k$ anyway. So $A$ maps the whole of $\mathcal{K}_k$ into itself:

$$
A\, \mathcal{K}_k \subseteq \mathcal{K}_k .
$$

$\mathcal{K}_k$ is **invariant** under $A$. Then $A^k b \in \mathcal{K}_k$, so $\mathcal{K}_{k+1} = \mathcal{K}_k$, and the same holds at every later step: the Krylov space has stopped growing for good. In the Arnoldi relation, the last row of $\bar H_k$ is $(0, \dots, 0, h_{k+1,k}) = 0$, so it drops out:

$$
A V_k = V_k H_k .
$$

So $H_k$ is $A$ itself, acting on $\mathcal{K}_k$ and written in the basis $V_k$: for every $y \in \mathbb{R}^k$, $A\, (V_k y) = V_k\, (H_k y)$.

**When it happens.** In exact arithmetic, the step at which the Krylov space stops growing depends on $A$ and $b$ together. Call it $d$: the first $k$ for which $A^k b$ is a combination of $b, Ab, \dots, A^{k-1} b$. Writing that combination out,

$$
A^d b + a_{d-1} A^{d-1} b + \cdots + a_1 A b + a_0\, b = 0
$$

for some numbers $a_0, \dots, a_{d-1}$. This says that the polynomial

$$
\mu_b(z) = z^d + a_{d-1} z^{d-1} + \cdots + a_1 z + a_0
$$

satisfies $\mu_b(A)\, b = 0$, and that no polynomial of lower degree (with leading coefficient 1) does. $\mu_b$ is the **minimal polynomial of $b$** (with respect to $A$), and $d$ is called the **grade** of $b$. Cayley–Hamilton (Step 1.2) gives $d \le n$. Every polynomial $p$ with $p(A)\, b = 0$ is a multiple of $\mu_b$ (divide $p$ by $\mu_b$; the remainder has lower degree and also sends $b$ to zero, so it is zero), so $p$ is zero wherever $\mu_b$ is.

For a diagonalisable $A$ there is a concrete way to find $d$. Write $b$ in a basis of eigenvectors $u_i$ of $A$, with $A u_i = \lambda_i u_i$: $b = \sum_i \gamma_i u_i$ for some coefficients $\gamma_i$. Then $p(A)\, b = \sum_i \gamma_i\, p(\lambda_i)\, u_i$ for any polynomial $p$, which is zero exactly when $p(\lambda_i) = 0$ for every $i$ with $\gamma_i \ne 0$. Say that **$b$ touches** an eigenvalue $\lambda$ if $b$ has a nonzero component $\gamma_i$ along some eigenvector $u_i$ that belongs to $\lambda$. Then

$$
\mu_b(z) = \prod_{\lambda \text{ touched by } b} (z - \lambda), \qquad
d = \text{the number of distinct eigenvalues that } b \text{ touches}.
$$

An eigenvalue that appears several times counts once, and eigenvalues whose eigenvectors $b$ misses don't count at all. (A matrix that isn't diagonalisable, like `nilpotent2`, still has a $\mu_b$; it just can't be read off this way.)

**What $H_d$ is.** At the breakdown, $H_d$ is $A$ acting on $\mathcal{K}_d$, and its eigenvalues are exactly the roots of $\mu_b$. (This is a standard fact: $\mathcal{K}_d$ is generated by the single vector $b$, and then the characteristic polynomial of $A$ on it is $\mu_b$.) For a diagonalisable $A$, the eigenvalues of $H_d$ are the eigenvalues $b$ touches.

**Good or bad.** Step 1.2 used Cayley–Hamilton to show that $A^{-1} b$ lies in $\mathcal{K}_n$. The same argument with $\mu_b$ goes further. Move the $a_0\, b$ term of $\mu_b(A)\, b = 0$ to the other side:

$$
A \big( A^{d-1} + a_{d-1} A^{d-2} + \cdots + a_1 I \big)\, b = -a_0\, b .
$$

- **If $a_0 = \mu_b(0) \ne 0$**, divide by $-a_0$. The vector $x_\ast = -\frac{1}{a_0} \big( A^{d-1} + \cdots + a_1 I \big)\, b$ lies in $\mathcal{K}_d$ and solves $A x_\ast = b$. GMRES picks the smallest residual over $\mathcal{K}_d$, which is now 0, so **$x_d$ is exact**. Nothing here needs $A$ to be nonsingular.
- **If $\mu_b(0) = 0$**, there is nothing to divide by, and in fact **no vector of $\mathcal{K}_d$ solves $Ax = b$**. If $x = q(A)\, b$ did, for some polynomial $q$, then $(A\, q(A) - I)\, b = 0$, so the polynomial $z\, q(z) - 1$ would be zero wherever $\mu_b$ is, including at $z = 0$. But at $z = 0$ it is $-1$. Since $\mathcal{K}_d$ never grows, GMRES can never reach $b$.

For a diagonalisable $A$, $\mu_b(0) = 0$ means that **$b$ touches the eigenvalue 0**: it has a component along a null vector of $A$ (a vector $u \ne 0$ with $Au = 0$). A nonsingular $A$ has no eigenvalue 0, so for it every breakdown is good.

`eigenvalues_touched_by` below finds the eigenvalues $b$ touches, by writing $b$ in the eigenvector basis that `np.linalg.eig` returns. The check runs `arnoldi` for up to $n$ steps, lets it stop at its breakdown, and compares the eigenvalues of $H_d$ with those $b$ touches, on three of the examples.

In [ ]:
def eigenvalues_touched_by(A, b, *, threshold=1e-8):
    """The distinct eigenvalues of a diagonalisable A whose eigenvectors carry part of b.

    Writes b = X gamma in the basis X of eigenvectors and keeps every eigenvalue
    whose part of b, |gamma_i| ||x_i||, is above threshold * ||b||. Eigenvalues
    closer together than threshold * max |lambda| count as one.
    """
    eigenvalues, X = np.linalg.eig(A)
    gamma = np.linalg.solve(X, b)
    carried = np.abs(gamma) * np.linalg.norm(X, axis=0) > threshold * np.linalg.norm(b)
    scale = np.abs(eigenvalues).max()
    distinct = []
    for eigenvalue in eigenvalues[carried]:
        if all(abs(eigenvalue - kept) > threshold * scale for kept in distinct):
            distinct.append(eigenvalue)
    return np.sort(np.real_if_close(np.array(distinct)))


for name, A, b in [("eig_rhs", A_eig, b_eig), ("convdiff_n, c = 0, b = ones", A_cd0, b_cd0),
                   ("singular3", A_sing, b_sing)]:
    touched = eigenvalues_touched_by(A, b)
    V, H_bar, _ = arnoldi(CountingMatvec(A), b, b.shape[0])   # stops by itself at the breakdown
    d = H_bar.shape[1]
    H = H_bar[:d, :d]                                          # H_d, the top d x d block
    print(f"{name}:  b touches {len(touched)} of the {b.shape[0]} eigenvalues;  Arnoldi stopped after "
          f"{d} steps with h_(d+1,d) = {H_bar[d, d - 1]}")
    print(f"   eigenvalues b touches   {touched}")
    print(f"   eigenvalues of H_{d}     {np.sort(np.linalg.eigvals(H).real)}")
    print(f"   ||A V_d - V_d H_d|| = {np.linalg.norm(A @ V[:, :d] - V[:, :d] @ H):.1e}")

On all three, Arnoldi stops by itself after exactly $d$ steps, $d$ being the number of eigenvalues $b$ touches, and the eigenvalues of $H_d$ are those eigenvalues, to round-off. `convdiff_n` with $c = 0$ and $b = (1, \dots, 1)$ touches only 10 of its 20 eigenvalues; section 5.3 explains why. `singular3`'s $b$ touches all three eigenvalues of $A$, including 0, and so $H_3$ has the eigenvalue 0, to round-off: it is singular.

**What the rotation pair does at a breakdown.** At step $k = d$ the subdiagonal entry is $h_{k+1,k} = 0$, so the rotation pair is $(\tilde h_{kk},\ 0)$, and

$$
R_{kk} = |\tilde h_{kk}|, \qquad s_k = 0, \qquad c_k = \pm 1 .
$$

$G_k$ is the identity, up to a sign, and everything depends on the single number $\tilde h_{kk}$, which decides whether $H_k$ is invertible. The earlier rotations $G_1, \dots, G_{k-1}$ only mix rows $1, \dots, k$, and row $k+1$ of $\bar H_k$ is zero. So the top $k \times k$ block of the rotated matrix is an orthogonal matrix times $H_k$, and $\det R_k = \pm \det H_k$. Before the breakdown every $h_{j+1,j}$ was nonzero, so every earlier pivot $R_{jj} \ge |h_{j+1,j}|$ is positive. Therefore, at a breakdown,

$$
H_k \text{ is singular} \iff R_{kk} = 0 \iff \tilde h_{kk} = 0 .
$$

The two entries of the rotation pair, each zero or not, give four cases:

| | $h_{k+1,k} \ne 0$ | $h_{k+1,k} = 0$: a breakdown |
|---|---|---|
| $\tilde h_{kk} \ne 0$ | an ordinary step: $0 < \lvert s_k \rvert < 1$, and the residual shrinks | **good breakdown**: $R_{kk} = \lvert \tilde h_{kk} \rvert > 0$, $s_k = 0$, and $x_k$ is exact (5.3) |
| $\tilde h_{kk} = 0$ | **stagnation**: $c_k = 0$ and $\lvert s_k \rvert = 1$, so no progress, but $R_{kk} = \lvert h_{k+1,k} \rvert > 0$ and the Krylov space keeps growing. Not a breakdown (`shift_n`, Step 4.7) | **bad breakdown**: $R_{kk} = 0$ and $R_k$ is singular (5.4) |

The right-hand column is the subject of the next two sections.

### 5.3 Good breakdowns: the Krylov space holds the answer

Here the rotation pair is $(\tilde h_{kk},\ 0)$ with $\tilde h_{kk} \ne 0$. $G_k$ is the identity up to a sign, the pivot $R_{kk} = |\tilde h_{kk}|$ is positive, so $R_k$ is invertible, $|g_{k+1}| = 0$ really is the smallest residual, and $x_k$ is exact. This is the case `gmres_v3` already gets right.

**For a nonsingular $A$, every breakdown is good** (Saad, *Iterative Methods for Sparse Linear Systems*, 2nd ed., Prop. 6.10). Section 5.2 showed this through $\mu_b(0)$. There is also a one-line proof: if $H_k z = 0$ for some $z$, then $A\, (V_k z) = V_k H_k z = 0$, so $V_k z = 0$ because $A$ is nonsingular, so $z = 0$ because the columns of $V_k$ are independent. So $H_k$ is invertible, and with it $R_k$.

What decides *when* a good breakdown comes is how many eigenvalues $b$ touches. `rotation_pairs` below runs the Arnoldi step and the Givens update exactly as `gmres_v3` does, until a breakdown, and records each step's rotation pair, its pivot, and the length of its Hessenberg column (which equals $\|A v_k\|_2$; section 5.9 needs it). The table runs it on matrices of several kinds, next to the number of eigenvalues $b$ touches and the true residual of `gmres_v3`'s answer. The solver runs with $\text{rtol} = 0$, so that only the breakdown stops it.

In [ ]:
def rotation_pairs(A, b, max_steps=None):
    """Run Arnoldi and gmres_v3's Givens update until a breakdown, recording each rotation pair.

    For looking at the numbers only. Step k builds the rotation G_k from the pair
    (h-tilde_kk, h_{k+1,k}): the diagonal entry of column k of H-bar after the
    earlier rotations, and the subdiagonal entry Arnoldi computed.

    Returns:
        dict with one list entry per step k = 1, 2, ...:
            "diagonal"     h-tilde_kk,
            "subdiagonal"  h_{k+1,k} (0 at a flagged breakdown),
            "pivot"        R_kk, the length of the pair,
            "column_norm"  the length of column k of H-bar, equal to ||A v_k||,
        and "steps" (the step that broke down, or the last one run), "breakdown",
        "R" (R_k) and "g" (the rotated right-hand side, length k + 1).
    """
    n = b.shape[0]
    if max_steps is None:
        max_steps = n
    V = np.zeros((n, max_steps + 1))
    V[:, 0] = b / np.linalg.norm(b)
    R = np.zeros((max_steps, max_steps))
    g = np.zeros(max_steps + 1)
    g[0] = np.linalg.norm(b)
    rotations = []
    trace = {"diagonal": [], "subdiagonal": [], "pivot": [], "column_norm": []}
    for k in range(1, max_steps + 1):
        h, breakdown = arnoldi_step_classical_gram_schmidt_twice(lambda v: A @ v, V, k - 1)
        trace["subdiagonal"].append(h[k])
        trace["column_norm"].append(np.linalg.norm(h))
        for i, (c_i, s_i) in enumerate(rotations):
            h[i], h[i + 1] = c_i * h[i] + s_i * h[i + 1], -s_i * h[i] + c_i * h[i + 1]
        radius, cosine, sine = givens_rotation(h[k - 1], h[k])
        rotations.append((cosine, sine))
        R[: k - 1, k - 1] = h[: k - 1]
        R[k - 1, k - 1] = radius
        g[k - 1], g[k] = cosine * g[k - 1], -sine * g[k - 1]
        trace["diagonal"].append(h[k - 1])
        trace["pivot"].append(radius)
        if breakdown:
            break
    trace.update(steps=k, breakdown=breakdown, R=R[:k, :k], g=g[: k + 1])
    return trace


good_examples = [
    ("eig_rhs", A_eig, b_eig),
    ("convdiff_n, c = 0, b = ones", A_cd0, b_cd0),
    ("diag(1, 1, 2, 2, 3, 3)", A_repeated, b_repeated),
    ("geo3", A_geo3, b_geo3),
    ("diag_spread", A_spread, b_spread),
    ("convdiff_n, c = 20", A_cd, b_cd),
    ("singular3, b = (1, 1, 0)", A_sing, b_in_range),
]
print(f"{'example':28s} {'n':>3s} {'b touches':>10s} {'breaks at k':>12s}   {'h-tilde_kk':>10s} {'h_k+1,k':>8s}"
      f" {'R_kk':>10s}   {'true ||b - A x|| / ||b||':>24s}")
for name, A, b in good_examples:
    trace = rotation_pairs(A, b)
    x, record = gmres_v3(CountingMatvec(A), b, rtol=0.0)
    print(f"{name:28s} {b.shape[0]:3d} {len(eigenvalues_touched_by(A, b)):10d} {trace['steps']:12d}   "
          f"{trace['diagonal'][-1]:10.4g} {trace['subdiagonal'][-1]:8.0f} {trace['pivot'][-1]:10.4g}   "
          f"{np.linalg.norm(b - A @ x) / np.linalg.norm(b):24.1e}")
print("\nsingular3 with b = (1, 1, 0): b touches", eigenvalues_touched_by(A_sing, b_in_range),
      "  gmres_v3 returns x =", gmres_v3(CountingMatvec(A_sing), b_in_range)[0])

Each one breaks down at $k = d$, the number of eigenvalues $b$ touches, with a rotation pair $(\tilde h_{kk},\ 0)$ whose first entry is nowhere near zero, and each answer is exact up to round-off. What differs between the kinds of matrix is only what makes $d$ small or large:

- **$b$ is an eigenvector** (`eig_rhs`). $b$ touches one eigenvalue, so $d = 1$. The pair is $(2, 0)$: with no earlier rotations, $\tilde h_{11} = h_{11} = v_1^\mathsf{T} A v_1 = 2$, the eigenvalue, and $x_1 = b/2$.
- **$b$ shares a symmetry of $A$** (`convdiff_n` with $c = 0$ and $b = (1, \dots, 1)$): $d = 10 = n/2$. Reversing the grid, $s_i \mapsto s_{n+1-i}$, leaves the symmetric matrix of $-u''$ unchanged, and it leaves $b = (1, \dots, 1)$ unchanged too. So it leaves $Ab, A^2 b, \dots$ unchanged: every Krylov vector is symmetric about the middle of the grid, and such vectors form a space of dimension $n/2$. In terms of eigenvectors, those of this matrix are $\sin(j \pi s_i)$ for $j = 1, \dots, 20$, symmetric for odd $j$ and antisymmetric for even $j$, and $b$ touches only the 10 symmetric ones. With $c = 20$ the advection term spoils the symmetry, and the same $b$ touches all 20.
- **$A$ has repeated eigenvalues** ($\operatorname{diag}(1, 1, 2, 2, 3, 3)$). $b$ has a component along all six eigenvectors but touches only three distinct eigenvalues, so $d = 3$: $(A - I)(A - 2I)(A - 3I)\, b = 0$.
- **Nothing special** (`geo3`, `diag_spread`, `convdiff_n` with $c = 20$). $b$ touches all $n$ eigenvalues and $d = n$. These are the breakdowns that Steps 2–4 flagged at $k = n$, where the Krylov space is all of $\mathbb{R}^n$.
- **$A$ singular, $b$ in its range** (`singular3` with $b = (1, 1, 0)$). $b$ touches the eigenvalues 1 and 2 but not 0, so $d = 2$, $\mu_b(z) = (z - 1)(z - 2)$ has $\mu_b(0) = 2 \ne 0$, and $x_2 = (1, \tfrac12, 0)$ is exact. A singular $A$ doesn't by itself make a breakdown bad.

Figure 5.1 shows three of them as residual histories, again with $\text{rtol} = 0$. An exact zero has no place on a log axis, so it is drawn on the bottom edge of the plot, as in Step 4.

In [ ]:
bottom = 1e-17   # where an exact zero is drawn
fig, ax = plt.subplots(figsize=(9, 4.2), constrained_layout=True)
for color, (name, A, b) in zip(SERIES_COLORS, [("diag(1, 1, 2, 2, 3, 3)", A_repeated, b_repeated),
                                               ("convdiff_n, c = 0", A_cd0, b_cd0),
                                               ("convdiff_n, c = 20", A_cd, b_cd)]):
    _, record = gmres_v3(CountingMatvec(A), b, rtol=0.0)
    b_norm = np.linalg.norm(b)
    d = len(eigenvalues_touched_by(A, b))
    plot_residuals(ax, np.maximum(record["residual_history"], bottom * b_norm), b_norm=b_norm, color=color,
                   label=f"{name}, n = {b.shape[0]}:  b touches {d}")
    print(f"{name:30s} breakdown at k = {record['iterations']:2d},  residual just before it: "
          f"{record['residual_history'][-2] / b_norm:.1e} ||b||")
ax.set_ylim(bottom / 2, 3)
ax.legend(frameon=False, loc="center left", bbox_to_anchor=(0.5, 0.45), fontsize=9,
          title="b = (1, ..., 1), eigenvalues b touches:", title_fontsize=9, alignment="left")
style_axes(ax, xticks=range(0, 21, 2))
add_figure_caption(fig, "5.1", r"Good breakdowns, with rtol = 0. Each residual drops to exactly 0 (bottom edge) at the step k = d where the Krylov space stops growing, d being the number of distinct eigenvalues b touches.")
plt.show()

Each residual drops to exactly zero at $k = d$, from wherever it was the step before: $0.13\, \|b\|_2$ on the repeated-eigenvalue matrix, $0.32\, \|b\|_2$ and $0.14\, \|b\|_2$ on the two `convdiff_n` problems. A good breakdown is GMRES finishing early. On `convdiff_n` with $c = 0$ it finds the exact solution of a 20×20 problem in 10 steps, because $b$'s symmetry halved the space it has to search.

### 5.4 Bad breakdowns: the Krylov space runs into the null space of $A$

Here the rotation pair is $(0, 0)$: both entries are zero, exactly or to round-off. `givens_rotation` returns radius 0 and the identity ($c_k = 1$, $s_k = 0$), so the pivot is $R_{kk} = 0$ and $R_k$ is singular.

**What the small problem says now.** The last row of $R_k$ is $(0, \dots, 0, R_{kk}) = 0$. In the rotated least-squares problem of section 4.1,

$$
\big\| g - \bar R_k\, y \big\|_2^2 = \sum_{i=1}^{k-1} \big( g_i - (R_k\, y)_i \big)^2 + g_k^2 + g_{k+1}^2 ,
$$

where $\bar R_k$ is $R_k$ with a zero row below it, the term $g_k^2$ is now out of reach of every $y$, just like $g_{k+1}^2$. The first $k - 1$ equations can still be met exactly, since the top-left $(k-1) \times (k-1)$ block of $R_k$ has positive pivots. $G_k$ is the identity, so it left $g_k$ as it was, $|g_k| = \|r_{k-1}\|_2$, and set $g_{k+1} = 0$. The smallest residual over $\mathcal{K}_k$ is therefore

$$
\min_y \big\| \beta e_1 - \bar H_k\, y \big\|_2 = \sqrt{g_k^2 + g_{k+1}^2} = |g_k| = \|r_{k-1}\|_2 .
$$

**Step $k$ made no progress**, and $|g_{k+1}| = 0$ isn't the residual at all: section 4.1's $\|r_k\|_2 = |g_{k+1}|$ needed the $k$-th equation to be solvable. The best vector in $\mathcal{K}_k$ is $x_{k-1}$, found at the previous step, and since $\mathcal{K}_k$ never grows again, $x_{k-1}$ is the best GMRES will ever find.

**What causes it.** By 5.2 a bad breakdown is $\mu_b(0) = 0$. Then $\mu_b(z) = z\, p(z)$ for a polynomial $p$ of degree $d - 1$. The vector $u = p(A)\, b$ is not zero (no polynomial of degree below $d$ sends $b$ to zero), it lies in $\mathcal{K}_d$, and $A u = \mu_b(A)\, b = 0$: **the Krylov space contains a null vector of $A$**, and the breakdown comes at the step that adds it. For a diagonalisable $A$ that is the case exactly when $b$ touches the eigenvalue 0.

The two new examples show two ways it happens. For each, the rotation pair at the breakdown, and on `singular3` every iterate $x_k$ that `gmres_v3` produces when stopped after $k$ steps, with its true residual. The last line solves `singular3`'s small problem at $k = 3$ with `lstsq` instead of the Givens update. `lstsq` treats a singular value below round-off as zero, so it gives the minimum derived above.

In [ ]:
for name, A, b in [("singular3", A_sing, b_sing), ("nilpotent2", A_nil, b_nil)]:
    trace = rotation_pairs(A, b)
    print(f"{name}: breaks down at k = {trace['steps']};  pair (h-tilde_kk, h_k+1,k) = "
          f"({trace['diagonal'][-1]:.2e}, {trace['subdiagonal'][-1]:.0f});  pivot R_kk = {trace['pivot'][-1]:.2e}")
print("singular3: b touches", eigenvalues_touched_by(A_sing, b_sing), "   nilpotent2: A b =", A_nil @ b_nil)

print("\nsingular3, gmres_v3 stopped after k steps:")
for k in (1, 2, 3):
    x, record = gmres_v3(CountingMatvec(A_sing), b_sing, rtol=0.0, max_iterations=k)
    print(f"  k = {k}:  x_k = {x},  |g_k+1| = {record['residual_history'][-1]:.4f},  "
          f"true ||b - A x_k|| = {np.linalg.norm(b_sing - A_sing @ x):.4f}")
print("  best possible: ||(0, 0, 1)|| = 1")

_, H_bar_3, beta = arnoldi(CountingMatvec(A_sing), b_sing, 3)
beta_e1 = np.array([beta, 0.0, 0.0, 0.0])
y, *_ = np.linalg.lstsq(H_bar_3, beta_e1, rcond=None)
print(f"\nsingular3, k = 3:  lstsq gives min ||beta e_1 - H-bar_3 y|| = {np.linalg.norm(beta_e1 - H_bar_3 @ y):.4f}")

- **`singular3`: no solution exists.** $b = (1, 1, 1)$ touches all three eigenvalues 1, 2 and 0, so $d = 3$, with $\mu_b(z) = (z - 1)(z - 2)\, z$. The range of $A$ is the $xy$-plane, so the best any $x$ can do is to match $b$'s first two entries and leave the residual $(0, 0, 1)$, of norm 1. GMRES gets there at step 2: $x_2 = (1, \tfrac12, \tfrac32)$ has exactly that residual. (Its third entry doesn't matter, since $A$ sets it to zero.) Step 3 adds the null direction $e_3$, which can't help, so the step should report no progress, and `lstsq` on the small problem confirms the minimum is 1.0. In exact arithmetic the pair at step 3 would be $(0, 0)$. In floating point $\tilde h_{33} = -1.7 \times 10^{-16}$, round-off instead of zero, and `gmres_v3`'s back-substitution divides $g_3$, of size about 1, by $R_{33} = 1.7 \times 10^{-16}$. Hence the $-1.47 \times 10^{16}$ in $x_3$, and a true residual of 2.24.
- **`nilpotent2`: a solution exists but can't be reached.** $Ab = 0$, so $\mu_b(z) = z$ and $d = 1$: $\mathcal{K}_1 = \operatorname{span}\{e_1\}$ is the null space of $A$ itself. The solution $x = e_2$ is not in it, and $\mathcal{K}_1$ never grows. Here $A v_1$ is exactly zero, so the pair is exactly $(0, 0)$, $R_{11} = 0$, and `gmres_v3` divides by zero. The best vector in $\mathcal{K}_1$ is $x_0 = 0$. A bad breakdown is a property of $b$'s Krylov space, not only of whether $Ax = b$ has a solution.

Compare `singular3` with $b = (1, 1, 0)$ in 5.3: the same matrix, and a good breakdown. Whether a breakdown is good or bad depends on $b$ as much as on $A$.

Figure 5.2 shows `singular3`'s residuals on a linear axis, where a zero can be drawn as a zero. Figure 5.3 shows the triangular factor $R_k$ at the breakdown for `singular3` and `nilpotent2`, next to `geo3`'s good one, on a log scale shared by the three panels. Blank cells are exact zeros.

In [ ]:
b_norm = np.linalg.norm(b_sing)
_, record_sing = gmres_v3(CountingMatvec(A_sing), b_sing)
true_residuals = [b_norm]   # x_0 = 0
for k in (1, 2, 3):
    x, _ = gmres_v3(CountingMatvec(A_sing), b_sing, rtol=0.0, max_iterations=k)
    true_residuals.append(np.linalg.norm(b_sing - A_sing @ x))

fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
steps = np.arange(4)
ax.plot(steps, np.array(record_sing["residual_history"]) / b_norm, color=SERIES_COLORS[0], linewidth=2,
        marker="o", markersize=6, label=r"gmres_v3 reports: $|g_{k+1}|$")
ax.plot(steps, np.array(true_residuals) / b_norm, color=SERIES_COLORS[1], linestyle="none", marker="x",
        markersize=10, markeredgewidth=2, label=r"true residual $\|b - A x_k\|_2$ of gmres_v3's $x_k$")
ax.axhline(1.0 / b_norm, color=REFERENCE_COLOR, linewidth=1, linestyle="--")
ax.text(0.05, 1.0 / b_norm + 0.03, r"best possible: $\|(0, 0, 1)\|_2$", color=REFERENCE_COLOR, fontsize=9)
ax.set_xlabel("step k")
ax.set_ylabel(r"residual $/\ \|b\|_2$")
ax.set_ylim(-0.05, 1.45)
ax.set_title("singular3: A = diag(1, 2, 0), b = (1, 1, 1)")
ax.legend(frameon=False, loc="lower left")
style_axes(ax, xticks=steps)
add_figure_caption(fig, "5.2", r"gmres_v3 on singular3. The residual reaches the best possible value at k = 2. At the breakdown, k = 3, gmres_v3 reports 0 while its $x_3$ has a true residual larger than that of $x_0 = 0$.")
plt.show()

In [ ]:
panels = [("geo3", A_geo3, b_geo3), ("singular3", A_sing, b_sing), ("nilpotent2", A_nil, b_nil)]
factors = [rotation_pairs(A, b)["R"] for _, A, b in panels]
smallest = min(np.abs(R[R != 0]).min() for R in factors if np.any(R != 0)) / 10
largest = max(np.abs(R).max() for R in factors) * 1.5

fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), constrained_layout=True, gridspec_kw={"width_ratios": [3, 3, 1.6]})
for ax, (name, _, _), R in zip(axes, panels, factors):
    k = R.shape[0]
    image = plot_matrix(ax, R, log=True, vmin=smallest, vmax=largest, colorbar=False,
                        title=rf"{name}: $R_{k}$")
    ax.set_xticks(range(1, k + 1))
    ax.set_yticks(range(1, k + 1))
    pivot = R[k - 1, k - 1]
    ax.text(k, k, "0" if pivot == 0 else f"{pivot:.1e}", ha="center", va="center", fontsize=9,
            color="white" if pivot > 1e-3 else "#0b0b0b")
    if ax is not axes[0]:
        ax.set_ylabel("")
fig.colorbar(image, ax=axes, shrink=0.8, label="|entry|")
add_figure_caption(fig, "5.3", r"The triangular factor $R_k$ at the breakdown step k, with the pivot $R_{kk}$ printed in its cell. geo3's pivot is of the size of the other entries (good). singular3's is round-off, 16 orders of magnitude smaller, and nilpotent2's is exactly 0 (blank): $R_k$ is singular.")
plt.show()

In Figure 5.2 the residual drops from 1.73 to 1.0, the best possible, by step 2, and then `gmres_v3` reports 0 at step 3 while the true residual of its $x_3$ jumps to 2.24. In Figure 5.3, `geo3`'s pivot, 1.71, is as large as any other entry of its $R_3$; `singular3`'s is $1.7 \times 10^{-16}$, sixteen orders of magnitude below the rest, and `nilpotent2`'s is an exact zero. The two bad pivots differ only in how round-off treated them: on `nilpotent2` every number involved is an exact integer, on `singular3` the basis vectors $v_2$ and $v_3$ are not. Section 5.7 needs a different check for each.

### 5.5 Which matrices give which breakdowns

Sections 5.2–5.4 in one table. Each row is a kind of matrix and right-hand side, with the step $d$ at which the Krylov space stops growing, the rotation pair $(\tilde h_{dd},\ h_{d+1,d})$ at that step, and an example from this notebook.

| $A$ | $b$ | breaks down at | pair at that step | kind | example |
|---|---|---|---|---|---|
| any | an eigenvector, $Ab = \lambda b$ with $\lambda \ne 0$ | $d = 1$ | $(\lambda,\ 0)$ | good | `eig_rhs` |
| has a symmetry | shares it | $d$ = the eigenvalues $b$ touches, e.g. $n/2$ | $(\ne 0,\ 0)$ | good | `convdiff_n`, $c = 0$, $b$ = ones |
| repeated eigenvalues | anything | $d \le$ the number of distinct eigenvalues | $(\ne 0,\ 0)$ | good | $\operatorname{diag}(1, 1, 2, 2, 3, 3)$ |
| nonsingular, nothing special | nothing special | $d = n$ | $(\ne 0,\ 0)$ | good | `geo3`, `diag_spread`, `convdiff_n`, $c = 20$ |
| singular | touches only nonzero eigenvalues | $d$ = the eigenvalues $b$ touches | $(\ne 0,\ 0)$ | good | `singular3`, $b = (1, 1, 0)$ |
| singular | touches the eigenvalue 0 | $d$ = the eigenvalues $b$ touches | $(0,\ 0)$, in practice round-off | **bad** | `singular3`, $b = (1, 1, 1)$ |
| singular | a null vector itself, $Ab = 0$ | $d = 1$ | $(0,\ 0)$, here exactly | **bad** | `nilpotent2` |
| any | $A \mathcal{K}_k \perp b$ | (no breakdown) | $(0,\ \ne 0)$: stagnation, $\lvert s_k \rvert = 1$ | not a breakdown | `shift_n` |

The rule behind the table:

- **A nonsingular $A$ only ever has good breakdowns.** What $b$ looks like decides only *when* the breakdown comes: early if $b$ touches few eigenvalues, at $k = n$ otherwise.
- **A singular $A$ has a bad breakdown exactly when $b$'s Krylov space reaches a null vector of $A$**, which for a diagonalisable $A$ means that $b$ touches the eigenvalue 0. The same singular $A$ gives a good breakdown with a $b$ that avoids its null space.
- **When it is bad, round-off decides** whether the pivot comes out as an exact zero (`nilpotent2`) or as round-off (`singular3`), and with it which of `gmres_v4`'s two checks catches it (5.7).

In the simulation, $A$ is the Jacobian of an implicit time step, which is usually nonsingular, and then every breakdown is good news. A bad one means the linearised problem itself has no answer in the directions GMRES can reach: a singular Jacobian, or a preconditioner that is singular (Step 8).

### 5.6 Detecting a breakdown in floating point

In exact arithmetic a breakdown is $h_{k+1,k} = 0$. In floating point the remainder $w$ of $A v_k$ after orthogonalisation is never exactly zero: every product and subtraction leaves round-off of about $\varepsilon_{\text{machine}} \|A v_k\|_2$, where $\varepsilon_{\text{machine}} \approx 2.2 \times 10^{-16}$ is the spacing of float64 numbers near 1. So `normalise_remainder` (section 2.4) uses a relative test:

$$
\text{breakdown} \iff \|w\|_2 \le \varepsilon_{\text{machine}}\, \|A v_k\|_2 .
$$

A remainder below one unit of round-off in $A v_k$ carries no information, so this is the smallest threshold that makes sense. It works only if the orthogonalisation leaves no more than about $\varepsilon_{\text{machine}} \|A v_k\|_2$ behind at a true breakdown. Below, each of Step 2's three Arnoldi steps runs its own Arnoldi process up to a step where the Krylov space has run out in exact arithmetic: `geo3` at step 3 and `convdiff_n` ($c = 0$, $b$ = ones) at step 10. A step that is not flagged stores the remainder's length as $h_{j+1,j}$, so we can read it off, in units of $\varepsilon_{\text{machine}} \|A v_j\|_2$. Next to it is each run's loss of orthogonality $\|I - V_j^\mathsf{T} V_j\|_2$ at that point (Step 2.4).

In [ ]:
methods = [("classical Gram-Schmidt, one pass", arnoldi_step_classical_gram_schmidt),
           ("modified Gram-Schmidt", arnoldi_step_modified_gram_schmidt),
           ("classical Gram-Schmidt applied twice", arnoldi_step_classical_gram_schmidt_twice)]
for name, A, b, j in [("geo3", A_geo3, b_geo3, 3), ("convdiff_n, c = 0, b = ones", A_cd0, b_cd0, 10)]:
    print(f"{name}, step {j} (the Krylov space runs out here in exact arithmetic):")
    for method_name, step in methods:
        V, H_bar, _ = arnoldi(CountingMatvec(A), b, j, step=step)
        remainder = H_bar[j, j - 1]   # h_{j+1,j}: the remainder's length, or 0 if the step was flagged
        threshold = eps * np.linalg.norm(A @ V[:, j - 1])
        loss = np.linalg.norm(np.eye(j) - V[:, :j].T @ V[:, :j], 2)
        verdict = "flagged as a breakdown" if remainder == 0 else f"remainder {remainder / threshold:5.1f} eps ||A v_j||, missed"
        print(f"  {method_name:38s} {verdict:42s} loss of orthogonality {loss:.1e}")

With one pass, the remainder at a true breakdown is 11.7 and 13.0 times the threshold on `geo3`, and 23.3 and 26.9 times on `convdiff_n`, so the test misses all four. The reason is the loss of orthogonality next to them. By the time the Krylov space runs out, a one-pass basis is off by about $3 \times 10^{-15}$ on `geo3` and $10^{-14}$ on `convdiff_n`, and a pass of Gram–Schmidt against a basis that is off by that much leaves about that much of $A v_j$ behind (section 2.4). Two passes keep the basis orthonormal to a few $\varepsilon_{\text{machine}}$, leave less than the threshold, and the step is flagged. (Section 2.4 measured the two-pass remainder on `geo3`: $2.0 \times 10^{-31}$.) **The breakdown test relies on classical Gram–Schmidt applied twice.**

**A breakdown the test doesn't flag, harmlessly.** With $c = 0$ and $b_i = \sin(\pi s_i)$, $b$ is an eigenvector of $A$ in exact arithmetic, so $d = 1$. In floating point it isn't quite: $A$'s entries $2/\Delta s^2$ and $-1/\Delta s^2$ and the values $\sin(\pi s_i)$ are all rounded. So $A v_1$ is $\lambda v_1$ plus round-off, the remainder of step 1 is that round-off, and it is somewhat larger than the threshold.

In [ ]:
V_sine, H_sine, _ = arnoldi(CountingMatvec(A_cd0), b_cd0_sine, 1)
threshold = eps * np.linalg.norm(A_cd0 @ V_sine[:, 0])
print(f"sine, step 1:  remainder = {H_sine[1, 0] / threshold:.1f} eps ||A v_1||, so not flagged")
x, record = gmres_v3(CountingMatvec(A_cd0), b_cd0_sine)
print(f"gmres_v3 with the default rtol stops after {record['iterations']} step:  "
      f"||r_1|| = {record['residual_history'][-1] / np.linalg.norm(b_cd0_sine):.1e} ||b||,  converged = {record['converged']}")

The remainder is about 62 times the threshold, so step 1 is not flagged. It doesn't matter: the residual after step 1 is already $1.4 \times 10^{-14}\, \|b\|_2$, round-off, and the solver stops on its tolerance and reports convergence. That is how a missed *good* breakdown always looks, since its residual has already reached round-off. A missed one costs at most a few extra steps of noise, which is why the threshold can afford to be tight.

**Why it is written with `not`.** `normalise_remainder` computes `breakdown = not (norm_after > eps * norm_before)` rather than `norm_after <= eps * norm_before`. The two agree for every number, but not for NaN, because every comparison with NaN is false. If the matvec returns NaN, which in the simulation a Jacobian-vector product evaluated at an unphysical state can, the first form gives `breakdown = True` and the second `False`. So a NaN stops the Arnoldi process instead of being divided into the basis. Production's `arnoldi_step` uses the same form for the same reason.

### 5.7 `gmres_v4`: production's two checks

The bad breakdowns of 5.4 come in two forms, and production has a check for each.

**Check 1: is the step usable?** A step counts only if its Hessenberg column is finite and its pivot is positive, $R_{kk} > 0$. That catches an exact zero pivot (`nilpotent2`) and a NaN *before* anything is divided by them. An unusable step is not added to the factorisation: $R$, $g$ and the stored rotations stay as they were after step $k - 1$, and the iteration stops there. Back-substitution then gives $x_{k-1}$, which 5.4 showed is the best vector in $\mathcal{K}_k$.

**Check 2: is the answer any good?** A pivot of $1.7 \times 10^{-16}$ (`singular3`) is positive, so it passes check 1, and the step counts. So after every breakdown, and whenever the candidate $x$ is not finite, the solver doesn't trust $|g_{k+1}|$. It spends one more matvec on the true residual $\|b - Ax\|_2$ of its candidate, and keeps the candidate only if that residual is finite and no larger than $\|b\|_2$, the residual of the starting guess $x_0 = 0$. Otherwise it returns $x_0$. (Step 7, restarting, changes "the starting guess" to "the start of the cycle".) A good breakdown passes easily, since its true residual is round-off. `converged` is then judged by the true residual.

**The algorithm, `gmres_v4`.** As `gmres_v3` (section 4.5), with the two checks added:

1. Allocate `V` ($n \times (m+1)$), `R` ($m \times m$) and `g` (length $m + 1$) as zeros, with $m$ = `max_iterations`. Set $v_1 = b / \beta$, $g_1 = \beta$, and start an empty list of rotations.
2. For $k = 1, 2, \dots$:
   1. One Arnoldi step: it fills $v_{k+1}$, returns column $k$ of $\bar H$, and says whether it broke down. *(1 matvec.)*
   2. Apply the stored rotations $G_1, \dots, G_{k-1}$ to that column, and build $G_k$ from its rotation pair.
   3. **Check 1.** If the column is not finite or $R_{kk} = 0$, the step is unusable: stop, without it.
   4. Store $(c_k, s_k)$, write column $k$ of $R$, and rotate $g$. Now $|g_{k+1}|$ estimates $\|r_k\|_2$.
   5. Stop if $|g_{k+1}| \le \max(\text{rtol} \cdot \|b\|_2,\ \text{atol})$, or after a breakdown.
3. Solve $R_k\, y = g_{1:k}$ over the steps that counted, and form the candidate $x_k = V_k\, y$.
4. **Check 2.** After a breakdown, or if $x_k$ is not finite: compute $\|b - A x_k\|_2$ *(1 matvec)*, and keep $x_k$ only if that is finite and at most $\|b\|_2$; otherwise return $x_0 = 0$.

**Two new record keys.** The old keys keep their meaning: `iterations` counts the steps that were used, and `residual_history` holds $|g_{j+1}|$ for each of them.

- `breakdown`: whether the Krylov space stopped growing (or a step was unusable, or the candidate was not finite). Together with `converged` it tells a caller *why* the solver stopped (5.8).
- `residual_norm`: the residual norm of the $x$ that is returned. Without a breakdown it is $|g_{k+1}|$, the last entry of `residual_history`. After one it is the true residual just computed, or $\|b\|_2$ if $x_0$ was returned. This is production's `GMRESDiagnostics.residual_norm`.

`matvecs` counts every call, including an unusable step and the true-residual check.

In [ ]:
def gmres_v4(matvec, b, *, rtol=1e-10, atol=0.0, max_iterations=None, verbose=False):
    """GMRES with a Givens QR of H-bar, and production's two checks at a breakdown.

    Same iterates as gmres_v3 whenever no breakdown occurs. A step counts only
    if its Hessenberg column is finite and its pivot R_kk is nonzero. After a
    breakdown, or if the candidate x is not finite, the candidate's true
    residual is computed (one more matvec), and the candidate is kept only if
    that residual is finite and at most ||b||, the residual of x_0 = 0.
    Otherwise x_0 = 0 is returned.

    Args:
        matvec: function v -> A v. The solver never sees A itself.
        b: right-hand side, shape (n,).
        rtol: stop once ||r_k|| <= max(rtol * ||b||, atol).
        atol: absolute floor of that tolerance.
        max_iterations: largest k to try. Defaults to n.
        verbose: print the rotation pair and the residual at every step.

    Returns:
        x: the last usable iterate x_k, or x_0 = 0 if it failed the true-residual check.
        record: dict with
            "iterations"        number of steps k that were used,
            "matvecs"           number of matvec calls, including an unusable
                                step and the true-residual check,
            "residual_history"  [||r_0||, ||r_1||, ..., ||r_k||] as the Givens
                                update estimates them, |g_{j+1}|,
            "converged"         whether residual_norm met the tolerance,
            "breakdown"         whether the Krylov space stopped growing, a step
                                was unusable, or the candidate was not finite,
            "residual_norm"     the residual norm of the returned x: |g_{k+1}|,
                                or the true residual after a breakdown.
    """
    n = b.shape[0]
    if max_iterations is None:
        max_iterations = n
    b_norm = np.linalg.norm(b)
    beta = b_norm
    tolerance = max(rtol * b_norm, atol)

    # ----- k = 0: start from x_0 = 0, so the residual is b and v_1 = b / beta. -----
    residual_history = [b_norm]
    matvec_count = 0
    breakdown = False
    steps = 0                 # the steps used so far

    m = max_iterations
    V = np.zeros((n, m + 1))  # columns v_1, ..., v_{m+1}, filled one per step
    R = np.zeros((m, m))      # R_k, the triangular factor, filled one column per step
    g = np.zeros(m + 1)       # Omega_k beta e_1, the rotated right-hand side
    g[0] = beta
    rotations = []            # (c_j, s_j) of G_1, ..., G_k
    V[:, 0] = b / beta

    for k in range(1, max_iterations + 1):
        # 1. One Arnoldi step: v_{k+1}, column k of H-bar, and whether the Krylov space ran out.
        h, exhausted = arnoldi_step_classical_gram_schmidt_twice(matvec, V, k - 1)
        matvec_count += 1

        # 2. The earlier rotations G_1, ..., G_{k-1} on the new column, then G_k from its rotation pair.
        for i, (c_i, s_i) in enumerate(rotations):
            h[i], h[i + 1] = c_i * h[i] + s_i * h[i + 1], -s_i * h[i] + c_i * h[i + 1]
        radius, cosine, sine = givens_rotation(h[k - 1], h[k])

        if verbose:
            print(f"k = {k}:  rotation pair ({h[k - 1]:.3e}, {h[k]:.3e}),  pivot R_kk = {radius:.3e}")

        # 3. Check 1: a step with a non-finite column or a zero pivot is not used.
        usable = bool(np.all(np.isfinite(h))) and radius > 0.0
        if not usable:
            breakdown = True
            break

        # 4. Column k of R is final; G_k on the right-hand side gives the residual estimate.
        rotations.append((cosine, sine))
        R[: k - 1, k - 1] = h[: k - 1]
        R[k - 1, k - 1] = radius
        g[k - 1], g[k] = cosine * g[k - 1], -sine * g[k - 1]
        residual_history.append(abs(g[k]))
        steps = k
        breakdown = exhausted

        # 5. Stop once the estimate is small enough, or the Krylov space has stopped growing.
        if residual_history[-1] <= tolerance or breakdown:
            break

    # The candidate x_k: R_k y = g_{1:k} over the steps used, then x_k = V_k y.
    y = back_substitution(R[:steps, :steps], g[:steps])
    x = V[:, :steps] @ y

    if breakdown or not np.all(np.isfinite(x)):
        # Check 2: after a breakdown |g_{k+1}| can't be trusted, so measure the true residual.
        breakdown = True
        true_norm = np.linalg.norm(b - matvec(x))
        matvec_count += 1
        keep = bool(np.isfinite(true_norm)) and true_norm <= b_norm   # no worse than x_0 = 0
        if not keep:
            x = np.zeros_like(b)
            true_norm = b_norm
        residual_norm = true_norm
        converged = keep and true_norm <= tolerance
    else:
        residual_norm = residual_history[-1]
        converged = residual_norm <= tolerance

    record = {
        "iterations": steps,
        "matvecs": matvec_count,
        "residual_history": residual_history,
        "converged": bool(converged),
        "breakdown": breakdown,
        "residual_norm": residual_norm,
    }
    return x, record

On the examples of 5.1, plus `geo3` and `singular3` with $b = (1, 1, 0)$, both solvers with their default tolerance:

In [ ]:
for name, A, b in [("eig_rhs", A_eig, b_eig), ("geo3", A_geo3, b_geo3), ("singular3, b = (1, 1, 0)", A_sing, b_in_range),
                   ("singular3", A_sing, b_sing), ("nilpotent2", A_nil, b_nil)]:
    with np.errstate(divide="ignore", invalid="ignore"):   # gmres_v3 divides by zero on nilpotent2
        x3, record3 = gmres_v3(CountingMatvec(A), b)
        true3 = np.linalg.norm(b - A @ x3)
    x4, record4 = gmres_v4(CountingMatvec(A), b)
    print(f"{name}:")
    print(f"  gmres_v3:  converged = {record3['converged']!s:5s}                    reports {record3['residual_history'][-1]:.2g},"
          f"  true {true3:.3g},  {record3['matvecs']} matvecs,  x = {x3}")
    print(f"  gmres_v4:  converged = {record4['converged']!s:5s} breakdown = {record4['breakdown']!s:5s} "
          f" residual_norm {record4['residual_norm']:.3g},  true {np.linalg.norm(b - A @ x4):.3g},  "
          f"{record4['iterations']} iterations, {record4['matvecs']} matvecs,  x = {x4}")

- **The good breakdowns** (`eig_rhs`, `geo3`, `singular3` with $b = (1, 1, 0)$): the same $x$ as `gmres_v3`, `converged` and `breakdown` both true, and a `residual_norm` that is the true residual, round-off instead of an exact 0. Each costs one matvec more than in `gmres_v3`, for check 2.
- **`singular3`**: the pivot $1.7 \times 10^{-16}$ passes check 1, so all 3 steps count, and the candidate is `gmres_v3`'s $x_3$. Check 2 measures its true residual, 2.24, finds it larger than $\|b\|_2 = 1.73$, and returns $x_0 = 0$ instead. `converged` is false, `breakdown` true, and `residual_norm` is 1.73. The answer is safe, if not the best available: $x_2$ had residual 1.0 (section 5.9 comes back to this).
- **`nilpotent2`**: check 1 rejects step 1 (pivot exactly 0), so 0 iterations count and the candidate is $x_0 = 0$. Check 2 confirms its residual, $1 = \|b\|_2$. That is 2 matvecs for 0 iterations: the unusable step and the check. (Production's module docstring counts a solve as "iterations + cycles − 1" matvecs, plus one at a breakdown, which would be 1 here; it leaves out the call spent on an unusable step.)

**Checking against `gmres_v3`.** Away from breakdowns the two solvers do the same arithmetic, so `compare_versions` (stopping each after $k = 1, 2, \dots$ steps, with $\text{rtol} = 0$) should find identical iterates and identical residual histories. The only difference should come at the breakdown step $k = n$ of `geo3`, `diag_spread` and `convdiff_n`, where `gmres_v4` spends one more matvec on check 2.

In [ ]:
for name, A, b, ks in [("geo3", A_geo3, b_geo3, None),
                       ("diag_narrow", A_narrow, b_narrow, range(1, 12)),
                       ("diag_spread", A_spread, b_spread, None),
                       ("convdiff_n, c = 20", A_cd, b_cd, None)]:
    diff = compare_versions(gmres_v3, gmres_v4, A, b, ks=ks)
    differ = [int(k) for k, equal in zip(diff["k"], diff["matvecs_equal"]) if not equal]
    print(f"{name:20s} gmres_v3 vs gmres_v4, k = 1..{diff['k'][-1]}:  largest x_difference {diff['x_difference'].max():.1e},  "
          f"largest residual_difference {diff['residual_difference'].max():.1e},  matvecs differ at k = {differ}")
    assert diff["x_difference"].max() == 0.0 and diff["residual_difference"].max() == 0.0
    assert differ == ([] if ks is not None else [b.shape[0]])

_, record3 = gmres_v3(CountingMatvec(A_spread), b_spread, rtol=0.0)
_, record4 = gmres_v4(CountingMatvec(A_spread), b_spread, rtol=0.0)
print(f"\ndiag_spread, rtol = 0:  gmres_v3 {record3['matvecs']} matvecs, converged = {record3['converged']};   "
      f"gmres_v4 {record4['matvecs']} matvecs, converged = {record4['converged']}, breakdown = {record4['breakdown']}, "
      f"residual_norm = {record4['residual_norm']:.1e}")

The iterates and residual histories are identical, not just close, and the matvec counts differ only at the breakdown, by one. The last line shows one more difference, in what the solver *reports*. With $\text{rtol} = 0$, `gmres_v3` called the breakdown at $k = 20$ on `diag_spread` converged, since $|g_{21}| = 0 \le 0$. `gmres_v4` judges it by the true residual, $6.9 \times 10^{-13} = 1.5 \times 10^{-13}\, \|b\|_2$, which is not $\le 0$, so it reports `converged = False` with `breakdown = True`: the Krylov space ran out, and the answer is as good as round-off allows. With any tolerance above round-off, as in the table above for `geo3`, both flags are true.

### 5.8 Reading the record: `converged` and `breakdown`

The two flags together say why the solver stopped. One example of each combination, run through `gmres_v4`:

In [ ]:
outcomes = [
    ("diag_narrow", A_narrow, b_narrow, {}),
    ("eig_rhs", A_eig, b_eig, {}),
    ("geo3", A_geo3, b_geo3, {}),
    ("singular3, b = (1, 1, 0)", A_sing, b_in_range, {}),
    ("singular3", A_sing, b_sing, {}),
    ("nilpotent2", A_nil, b_nil, {}),
    ("diag_spread, max_iterations = 10", A_spread, b_spread, {"max_iterations": 10}),
]
print(f"{'example':34s} {'converged':>9s} {'breakdown':>9s} {'iterations':>10s} {'matvecs':>7s} {'residual_norm / ||b||':>22s}")
for name, A, b, options in outcomes:
    x, record = gmres_v4(CountingMatvec(A), b, **options)
    print(f"{name:34s} {record['converged']!s:>9s} {record['breakdown']!s:>9s} {record['iterations']:10d} "
          f"{record['matvecs']:7d} {record['residual_norm'] / np.linalg.norm(b):22.1e}")

| | `breakdown = False` | `breakdown = True` |
|---|---|---|
| **`converged = True`** | The tolerance was met in the usual way (`diag_narrow`). | The Krylov space ran out and held the answer: a good breakdown (`eig_rhs`, `geo3`, `singular3` with $b = (1, 1, 0)$). |
| **`converged = False`** | The solver ran out of iterations before the tolerance (`diag_spread` with `max_iterations = 10`). More steps, or a preconditioner (Step 8), would help. | The Krylov space ran out *without* the answer: a bad breakdown (`singular3`, `nilpotent2`), or a matvec that returned NaN. More steps can't help. The returned $x$ is finite and no worse than $x_0 = 0$, and `residual_norm` is its true residual. |

`residual_norm` is always the residual of the $x$ actually returned. Production's caller uses the record this way: `newton_krylov.py` counts every linear solve with `converged` false as a GMRES failure, and the failures that also have `breakdown` true separately, as GMRES breakdowns. It still applies the correction in both cases, and leaves the judgement to its own convergence test. That is safe only because a bad breakdown now returns something finite and no worse than doing nothing.

### 5.9 A relative pivot test

On `singular3`, `gmres_v4` returns $x_0 = 0$, with residual 1.73, although $x_2$, with residual 1.0, was the best GMRES can do (5.4). The round-off pivot $R_{33} = 1.7 \times 10^{-16}$ got past check 1 because that check is *absolute*: $R_{kk} > 0$. Check 2 then saw only the candidate $x_3$, and the only fallback it has is $x_0$.

The breakdown test of 5.6 is relative, $\|w\|_2 \le \varepsilon_{\text{machine}} \|A v_k\|_2$, and check 1 can be made relative in the same way. The pivot is the length of the rotation pair, and the pair is part of column $k$ of $\bar H$, whose length is $\|A v_k\|_2$ (the rotations don't change lengths, and neither does $V_{k+1}$). So a pivot below $\varepsilon_{\text{machine}} \|A v_k\|_2$ is a zero pivot up to round-off. `gmres_v4_relative_pivot` is `gmres_v4` with that one line changed, marked `# changed`:

```python
usable = bool(np.all(np.isfinite(h))) and radius > 0.0                          # gmres_v4
usable = bool(np.all(np.isfinite(h))) and radius > eps * np.linalg.norm(h)      # gmres_v4_relative_pivot
```

An unusable step is dropped, so on `singular3` the candidate becomes $x_2$, which check 2 then accepts. Check 2 itself stays, as the backstop for whatever the threshold misses.

In [ ]:
def gmres_v4_relative_pivot(matvec, b, *, rtol=1e-10, atol=0.0, max_iterations=None, verbose=False):
    """gmres_v4 with a relative pivot test: a step counts only if R_kk > eps * ||column k of H-bar||.

    Identical to gmres_v4 except for the line marked "# changed". Same
    arguments and record. For this section only; the later steps build on
    gmres_v4, which matches production.
    """
    n = b.shape[0]
    if max_iterations is None:
        max_iterations = n
    b_norm = np.linalg.norm(b)
    beta = b_norm
    tolerance = max(rtol * b_norm, atol)

    residual_history = [b_norm]
    matvec_count = 0
    breakdown = False
    steps = 0

    m = max_iterations
    V = np.zeros((n, m + 1))
    R = np.zeros((m, m))
    g = np.zeros(m + 1)
    g[0] = beta
    rotations = []
    V[:, 0] = b / beta

    for k in range(1, max_iterations + 1):
        h, exhausted = arnoldi_step_classical_gram_schmidt_twice(matvec, V, k - 1)
        matvec_count += 1

        for i, (c_i, s_i) in enumerate(rotations):
            h[i], h[i + 1] = c_i * h[i] + s_i * h[i + 1], -s_i * h[i] + c_i * h[i + 1]
        radius, cosine, sine = givens_rotation(h[k - 1], h[k])

        if verbose:
            print(f"k = {k}:  rotation pair ({h[k - 1]:.3e}, {h[k]:.3e}),  pivot R_kk = {radius:.3e}")

        # Check 1, relative: the pivot must be more than round-off in ||A v_k|| = ||h||.
        usable = bool(np.all(np.isfinite(h))) and radius > np.finfo(float).eps * np.linalg.norm(h)  # changed
        if not usable:
            breakdown = True
            break

        rotations.append((cosine, sine))
        R[: k - 1, k - 1] = h[: k - 1]
        R[k - 1, k - 1] = radius
        g[k - 1], g[k] = cosine * g[k - 1], -sine * g[k - 1]
        residual_history.append(abs(g[k]))
        steps = k
        breakdown = exhausted

        if residual_history[-1] <= tolerance or breakdown:
            break

    y = back_substitution(R[:steps, :steps], g[:steps])
    x = V[:, :steps] @ y

    if breakdown or not np.all(np.isfinite(x)):
        breakdown = True
        true_norm = np.linalg.norm(b - matvec(x))
        matvec_count += 1
        keep = bool(np.isfinite(true_norm)) and true_norm <= b_norm
        if not keep:
            x = np.zeros_like(b)
            true_norm = b_norm
        residual_norm = true_norm
        converged = keep and true_norm <= tolerance
    else:
        residual_norm = residual_history[-1]
        converged = residual_norm <= tolerance

    record = {
        "iterations": steps,
        "matvecs": matvec_count,
        "residual_history": residual_history,
        "converged": bool(converged),
        "breakdown": breakdown,
        "residual_norm": residual_norm,
    }
    return x, record

Both versions on every example of this notebook, with the default tolerance. For each, whether the two returned the same $x$ and the same record, and the residual each one reports.

In [ ]:
every_example = [
    ("eig_rhs", A_eig, b_eig), ("geo3", A_geo3, b_geo3), ("singular3", A_sing, b_sing),
    ("singular3, b = (1, 1, 0)", A_sing, b_in_range), ("nilpotent2", A_nil, b_nil),
    ("diag(1, 1, 2, 2, 3, 3)", A_repeated, b_repeated), ("diag_narrow", A_narrow, b_narrow),
    ("diag_spread", A_spread, b_spread), ("diag_spread, 12 decades", A_spread12, b_spread12),
    ("convdiff_n, c = 0, b = ones", A_cd0, b_cd0), ("convdiff_n, c = 0, b = sine", A_cd0, b_cd0_sine),
    ("convdiff_n, c = 20", A_cd, b_cd), ("shift_n", A_shift, b_shift),
]
print(f"{'example':30s} {'same':>5s}   {'gmres_v4 residual_norm':>23s}   {'relative pivot residual_norm':>29s}")
for name, A, b in every_example:
    x_absolute, record_absolute = gmres_v4(CountingMatvec(A), b)
    x_relative, record_relative = gmres_v4_relative_pivot(CountingMatvec(A), b)
    same = np.array_equal(x_absolute, x_relative) and all(
        record_absolute[key] == record_relative[key] for key in record_absolute)
    print(f"{name:30s} {same!s:>5s}   {record_absolute['residual_norm']:23.3g}   {record_relative['residual_norm']:29.3g}")

x_relative, record_relative = gmres_v4_relative_pivot(CountingMatvec(A_sing), b_sing)
print(f"\nsingular3 with the relative test:  x = {x_relative},  iterations {record_relative['iterations']},  "
      f"matvecs {record_relative['matvecs']},  converged = {record_relative['converged']},  breakdown = {record_relative['breakdown']}")

The two agree exactly on every example but `singular3`, where the relative test drops step 3 and returns $x_2 = (1, \tfrac12, \tfrac32)$ with residual 1.0, the best possible, for the same 4 matvecs. `converged` is still false and `breakdown` still true, as they should be: no $x$ solves this system.

How close does the relative test come to dropping a legitimate step? Over every step of every example with only good breakdowns, the smallest ratio $R_{kk} / \|A v_k\|_2$, against the ratio of `singular3`'s round-off pivot:

In [ ]:
ratios = []
for name, A, b in every_example:
    if name in ("singular3", "nilpotent2"):
        continue
    trace = rotation_pairs(A, b)
    ratios.extend(np.array(trace["pivot"]) / np.array(trace["column_norm"]))
print(f"good examples, every step:  smallest R_kk / ||A v_k|| = {min(ratios):.3f}")

trace = rotation_pairs(A_sing, b_sing)
ratio = trace["pivot"][-1] / trace["column_norm"][-1]
print(f"singular3, step 3:          R_33 / ||A v_3|| = {ratio:.1e} = {ratio / eps:.2f} eps")

On the good examples the smallest ratio is about 0.16, fifteen orders of magnitude above $\varepsilon_{\text{machine}}$, so the relative test is nowhere near dropping a real step. A pivot below $\varepsilon_{\text{machine}} \|A v_k\|_2$ means $A$ is singular on $\mathcal{K}_k$ to working precision, and the step it would add is round-off anyway.

On the other side, `singular3`'s round-off pivot is $0.65\, \varepsilon_{\text{machine}} \|A v_3\|_2$: below the threshold, but not by much. A round-off zero can land on either side of $\varepsilon_{\text{machine}}$, just as the one-pass remainders of 5.6 did, which is why the relative test doesn't replace check 2 but sits in front of it. Whatever it misses, check 2 still makes safe, only with the fallback $x_0$ instead of $x_{k-1}$.

**What the tutorial keeps.** The later steps build on `gmres_v4`, with production's absolute test, so that Step 9 can compare the final NumPy version against production exactly. In production the relative test would be one line in `gmres.py: _arnoldi_steps`, a pivot `> eps * norm(column)` instead of `> 0`, and on a singular operator it would return the last useful iterate of the cycle instead of the cycle's starting point.

### Summary of Step 5

- A **breakdown**, $h_{k+1,k} = 0$, means $\mathcal{K}_k$ is invariant under $A$: the Krylov space has stopped growing for good, and $A V_k = V_k H_k$.
- It happens at step $d$, the degree of $b$'s minimal polynomial $\mu_b$. For a diagonalisable $A$, $d$ is the number of distinct eigenvalues $b$ touches. It comes early when $b$ is an eigenvector, shares a symmetry of $A$, or $A$ has repeated eigenvalues, and at $k = n$ otherwise.
- At a breakdown the rotation pair is $(\tilde h_{kk},\ 0)$, so $s_k = 0$ and $R_{kk} = |\tilde h_{kk}|$, and $H_k$ is singular exactly when $\tilde h_{kk} = 0$.
- **Good breakdown**: $\tilde h_{kk} \ne 0$, equivalently $\mu_b(0) \ne 0$, and $x_k$ is exact. For a nonsingular $A$ every breakdown is good (Saad, Prop. 6.10). A singular $A$ can have one too, when $b$ avoids its null space.
- **Bad breakdown**: $\tilde h_{kk} = 0$, equivalently the Krylov space contains a null vector of $A$. The step makes no progress, $|g_{k+1}| = 0$ means nothing, and $x_{k-1}$ is the best GMRES can ever find. In floating point the zero pivot is exact (`nilpotent2`) or round-off (`singular3`), and `gmres_v3` divides by it and reports convergence.
- The breakdown test is relative, $\|w\|_2 \le \varepsilon_{\text{machine}} \|A v_k\|_2$, and written with `not` so that NaN counts. It works only with classical Gram–Schmidt applied twice: one pass leaves 12–27 times the threshold at a true breakdown. A missed good breakdown is harmless, because its residual is already at round-off.
- **`gmres_v4`** adds production's two checks. Check 1: a step with a non-finite column or a zero pivot is not used. Check 2: after a breakdown, the true residual decides whether the candidate is kept or $x_0 = 0$ returned. New record keys `breakdown` and `residual_norm`. Without a breakdown its iterates are identical to `gmres_v3`'s. At one it spends one more matvec.
- `converged` and `breakdown` together say why the solver stopped: tolerance met, Krylov space held the answer, ran out of iterations, or Krylov space ran out without the answer.
- Check 1 is absolute, so `singular3`'s round-off pivot passes it, and `gmres_v4` falls back to $x_0$. A relative pivot test, $R_{kk} > \varepsilon_{\text{machine}} \|A v_k\|_2$, returns the best iterate $x_2$ there and changes nothing on the other examples.

**Ties to code.** Production handles breakdowns in three files of `WhistlerRegulatedHeatDispersion/src/time_integration/implicit/`:

- `arnoldi.py: arnoldi_step` returns the breakdown flag with the test of 5.6, `~(norm_after > eps * norm_before)`, and stores the new basis row as zeros. Its docstring notes that a NaN also reads as a breakdown.
- `givens.py: _rotation` returns the identity when both entries of the pair are zero, as `givens_rotation` does, and `fold`'s docstring says of a zero pivot that "whether such a step counts is the caller's decision".
- `gmres.py: _arnoldi_steps` is check 1: `usable = jnp.all(jnp.isfinite(column)) & (pivot > 0)`. An unusable step keeps the old factor (`jnp.where(usable, new, old)`) and isn't counted (`steps + usable`). `_restart_cycle` is check 2: `broke_down = arnoldi.breakdown | ~_is_finite(candidate)` chooses, with `lax.cond`, between `verified` (the true residual, and the candidate kept only if it is finite and no worse than at the start of the cycle) and `trusted` (the Givens estimate). `GMRESDiagnostics` documents `breakdown` as in 5.8. The module docstring's matvec count leaves out an unusable step (5.7).
- `newton_krylov.py` counts `gmres_failures` and `gmres_breakdowns` from these flags, and applies the correction either way.

**Next: Step 6.** `gmres_v4` completes Part I. It produces the same iterates as `gmres_v0` would in exact arithmetic, accurately in floating point (Steps 2 and 3), at a small cost per step (Step 4), and honestly at a breakdown. What it doesn't control is how many steps it needs. Step 4 found that the factor $|s_k|$ by which each step shrinks the residual is about 0.1 on `diag_narrow`, about 0.96 on `convdiff_n` and exactly 1 on `shift_n` until the last step. This step showed that the eigenvalues $b$ touches decide *when* GMRES finishes exactly. Step 6 shows that they also decide how fast it gets close: the residual is a polynomial in $A$ applied to $b$, and the eigenvalues decide how small that polynomial can be.